## Stage 1 — 트랙 수집 (YOLO + BotSort)

비디오에서 YOLO로 사람을 탐지하고 BotSort로 track_id를 부여합니다.

출력: `tracks/{영상이름}.pkl`

```python
{
  'tracks': {track_id: [{'frame': int, 'bbox': [x1,y1,x2,y2], 'conf': float}]},
  'frames_processed': int,
  'video_path': str,
  'git_commit': str,
  'git_branch': str,
}
```

- 1000프레임마다 체크포인트 자동 저장
- 재실행 시 중단된 프레임부터 자동 재개

In [ ]:
import sys as _sys, os as _os

IN_COLAB = 'google.colab' in _sys.modules or _os.environ.get('EYE_D_COLAB') == '1'

if IN_COLAB:
    if 'google.colab' in _sys.modules:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=True)
        import subprocess
        subprocess.run(['pip', 'install', '-q', 'ultralytics', 'boxmot'], check=True)


In [ ]:
import os, sys, cv2, numpy as np
from collections import defaultdict

IN_COLAB = globals().get('IN_COLAB', 'google.colab' in sys.modules or os.environ.get('EYE_D_COLAB') == '1')

if IN_COLAB:
    _drive_root = globals().get('DRIVE_PROJECT_ROOT', os.environ.get('EYE_D_DRIVE_ROOT',
        '/content/drive/MyDrive/projects/EYE-D/EYE-D'))
    edge_root = f'{_drive_root}/edge'
else:
    current_dir = os.path.abspath(os.getcwd())
    edge_root = os.path.abspath(os.path.join(current_dir, '..')) if os.path.basename(current_dir) == 'notebooks' else os.path.join(current_dir, 'edge')

if edge_root not in sys.path:
    sys.path.insert(0, edge_root)

import torch
DEVICE = '0' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}  |  edge_root: {edge_root}')


In [ ]:
import pathlib as _pl

# ── papermill -p 로 주입되는 파라미터 ────────────────────────────────────────
GIT_COMMIT  = "unknown"
GIT_BRANCH  = "unknown"

if IN_COLAB:
    VIDEO_PATH = os.path.join(globals().get('VIDEO_DIR', '/content/drive/MyDrive/EYE-D/data'), '14300002.avi')
    _tracks_dir = globals().get('TRACKS_DIR', '/content/drive/MyDrive/EYE-D/tracks')
else:
    VIDEO_PATH  = os.path.abspath(os.path.join(edge_root, '..', 'data', '14300002.avi'))
    _tracks_dir = os.path.abspath(os.path.join(edge_root, '..', 'tracks'))

MAX_FRAMES          = float('inf')
FRAME_STEP          = 1            # YOLO 실행 간격 (1=매 프레임, 5=5프레임마다)
YOLO_BATCH_SIZE     = 4            # 배치 YOLO 크기 (GPU 병렬 추론)
CONF_THRESH         = 0.40
MIN_BBOX_SIZE       = 40
TRACKS_DIR          = _tracks_dir
CHECKPOINT_INTERVAL = 1000
# ─────────────────────────────────────────────────────────────────────────────


In [ ]:
import subprocess as _sp

if GIT_COMMIT == "unknown":
    try:
        GIT_COMMIT = _sp.check_output(['git', 'rev-parse', 'HEAD'], cwd=edge_root, stderr=_sp.DEVNULL).decode().strip()
        GIT_BRANCH = _sp.check_output(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], cwd=edge_root, stderr=_sp.DEVNULL).decode().strip()
    except Exception:
        GIT_COMMIT, GIT_BRANCH = 'unknown', 'unknown'

print(f'git branch : {GIT_BRANCH}')
print(f'git commit : {GIT_COMMIT[:12]}...')


In [ ]:
TRACKS_PKL = str(_pl.Path(TRACKS_DIR) / f'{_pl.Path(VIDEO_PATH).stem}.pkl')

if not os.path.exists(VIDEO_PATH):
    raise FileNotFoundError(f'비디오 없음: {VIDEO_PATH}')

_cap = cv2.VideoCapture(VIDEO_PATH)
total_frames = int(_cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps          = _cap.get(cv2.CAP_PROP_FPS)
_cap.release()

_analyze = total_frames if MAX_FRAMES == float('inf') else int(min(MAX_FRAMES, total_frames))
print(f'비디오  : {os.path.basename(VIDEO_PATH)}')
print(f'전체    : {total_frames}프레임  |  {fps:.1f} FPS  |  {total_frames/fps:.0f}초')
print(f'처리 예정: {_analyze}프레임')
print(f'tracks  : {TRACKS_PKL}')

# 재개 여부 확인
_resume_from    = 0
_already_done   = False
if os.path.exists(TRACKS_PKL):
    import pickle as _pk
    with open(TRACKS_PKL, 'rb') as _f:
        _ckpt = _pk.load(_f)
    _frames_done = _ckpt.get('frames_processed', 0)
    if _frames_done >= _analyze:
        print(f'\n[완료] 이미 처리됨 ({_frames_done}프레임) — 수집 단계를 건너뜁니다.')
        _already_done = True
    else:
        _resume_from = _frames_done
        print(f'\n[재개] {_frames_done} / {_analyze} 프레임 처리됨 → 이어서 실행')


In [ ]:
try:
    from ultralytics import YOLO
    from boxmot.trackers.tracker_zoo import create_tracker
    TRACKER_AVAILABLE = True
    print('추론 라이브러리 로드 완료')
except ImportError as e:
    raise RuntimeError(f'Stage 1 실행에 ultralytics, boxmot 필요:\n  {e}')


def collect_tracks(video_path, max_frames=float('inf'), verbose=True,
                   start_frame=0, checkpoint_path=None, checkpoint_interval=1000,
                   existing_tracks=None, frame_step=1, yolo_batch_size=4):
    """
    YOLO + ByteTrack으로 track_id와 bbox를 수집합니다.

    최적화:
    - cap.grab(): 비YOLO 프레임은 픽셀 디코딩 없이 포인터만 이동
    - 배치 YOLO: yolo_batch_size 프레임을 묶어 한 번에 GPU 추론
    - 트래커는 항상 프레임 순서대로 업데이트 (정확도 보장)
    """
    import pickle as _ckpt_pickle, pathlib as _ckpt_pl, time as _time

    # ── 단계별 시간 측정 ────────────────────────────────────────────────────
    _timer = {'avi': 0, 'yolo': 0, 'track': 0, 'save': 0, 'total': _time.time()}

    half    = (DEVICE != 'cpu')
    detector = YOLO('yolov8n.pt')
    tracker  = create_tracker('bytetrack', reid_weights=None,
                               device=DEVICE, half=half)

    cap = cv2.VideoCapture(video_path)
    if start_frame > 0:
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
        if verbose:
            print(f'  프레임 {start_frame}부터 재개')

    track_data = defaultdict(list)
    if existing_tracks:
        for tid, recs in existing_tracks.items():
            track_data[tid].extend(recs)

    frame_idx  = start_frame
    _last_ckpt = start_frame
    _dummy     = None
    _t0        = _time.time()
    _progress  = 0
    _n_super   = max(1, frame_step) * max(1, yolo_batch_size)

    def _ckpt_save(fidx):
        _ckpt_pl.Path(checkpoint_path).parent.mkdir(parents=True, exist_ok=True)
        with open(checkpoint_path, 'wb') as _cf:
            _ckpt_pickle.dump({
                'tracks':           dict(track_data),
                'frames_processed': fidx,
                'video_path':       video_path,
                'git_commit':       GIT_COMMIT,
                'git_branch':       GIT_BRANCH,
            }, _cf)

    while cap.isOpened() and frame_idx < max_frames:
        # ── Phase 1: super-batch 수집 ─────────────────────────────────────────
        # 비YOLO 프레임: cap.grab() (헤더만 읽기, 픽셀 디코딩 생략)
        # YOLO 프레임:   cap.grab() + cap.retrieve() (완전 디코딩)
        collected = []

        for _ in range(_n_super):
            if frame_idx >= max_frames or not cap.isOpened():
                break
            if not cap.grab():
                break
            frame_idx += 1

            is_yolo = (frame_step <= 1 or frame_idx % frame_step == 0)
            if is_yolo:
                ret2, frame = cap.retrieve()
                if not ret2:
                    break
                if _dummy is None:
                    _dummy = np.zeros_like(frame)
                collected.append((frame_idx, frame, True))
            else:
                collected.append((frame_idx, None, False))

        if not collected:
            break

        if _dummy is None:
            h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            _dummy = np.zeros((h, w, 3), dtype=np.uint8)

        # ── Phase 2: 배치 YOLO 추론 ───────────────────────────────────────────
        yolo_items = [(idx, f) for idx, f, is_y in collected if is_y]
        yolo_map   = {}

        if yolo_items:
            batch_results = detector.predict(
                [f for _, f in yolo_items],
                conf=CONF_THRESH, classes=[0], verbose=False,
            )
            for (idx, _), res in zip(yolo_items, batch_results):
                yolo_map[idx] = res

        # ── Phase 3: 순서대로 트래커 업데이트 ────────────────────────────────
        for fidx, frame, is_yolo in collected:
            t_frame = frame if frame is not None else _dummy

            if is_yolo and fidx in yolo_map:
                res = yolo_map[fidx]
                if len(res.boxes) > 0:
                    dets   = res.boxes.data.cpu().numpy()
                    tracks = tracker.update(dets, t_frame)
                    if tracks is not None and len(tracks) > 0:
                        for t in tracks:
                            if t[5] < CONF_THRESH:
                                continue
                            bbox = [int(t[0]), int(t[1]), int(t[2]), int(t[3])]
                            w, h = bbox[2] - bbox[0], bbox[3] - bbox[1]
                            if w < MIN_BBOX_SIZE or h < MIN_BBOX_SIZE:
                                continue
                            track_data[int(t[4])].append({
                                'frame': fidx, 'bbox': bbox, 'conf': float(t[5]),
                            })
                else:
                    tracker.update(np.empty((0, 6)), t_frame)
            else:
                tracker.update(np.empty((0, 6)), t_frame)

            if checkpoint_path and (fidx - _last_ckpt) >= checkpoint_interval:
                _ckpt_save(fidx)
                _last_ckpt = fidx
                if verbose:
                    print(f'  [체크포인트] {fidx} 프레임 → {checkpoint_path}')

        _progress += len(collected)
        if verbose and _progress % 1000 < _n_super:
            _elapsed = int(_time.time() - _t0)
            print(f'  [진행] {frame_idx} 프레임  경과 {_elapsed}초')

    cap.release()
    n = sum(len(v) for v in track_data.values())
    if verbose:
        print(f'  완료 → 트랙 {len(track_data)}개  |  탐지 {n}개')
    # ── 실행 시간 요약 ────────────────────────────────────────────────────
    _elapsed_total = _time.time() - _timer['total']
    if verbose:
        print(f'  [시간] AVI {_timer["avi"]:.1f}초  YOLO {_timer["yolo"]:.1f}초  합계 {_elapsed_total:.1f}초')
        if _timer['yolo'] > 0:
            print(f'  [분석] YOLO가 {_timer["yolo"]/_elapsed_total*100:.1f}%의 시간 차지')

    return dict(track_data), frame_idx


print('collect_tracks 함수 정의 완료')

In [ ]:
import pickle as _pickle

if _already_done:
    print('[건너뜀] 이미 완료된 tracks.pkl 존재')
else:
    _existing = None
    if _resume_from > 0 and os.path.exists(TRACKS_PKL):
        with open(TRACKS_PKL, 'rb') as _f:
            _existing = _pickle.load(_f).get('tracks')

    _pl.Path(TRACKS_DIR).mkdir(parents=True, exist_ok=True)

    tracks, frames_processed = collect_tracks(
        VIDEO_PATH,
        max_frames=MAX_FRAMES,
        start_frame=_resume_from,
        checkpoint_path=TRACKS_PKL,
        checkpoint_interval=CHECKPOINT_INTERVAL,
        existing_tracks=_existing,
        frame_step=FRAME_STEP,
        yolo_batch_size=YOLO_BATCH_SIZE,
    )


In [ ]:
import pickle, pathlib

if _already_done:
    print('[건너뜀] 저장 단계 스킵')
else:
    out_path = pathlib.Path(TRACKS_PKL)
    with open(out_path, 'wb') as _f:
        pickle.dump({
            'tracks':           tracks,
            'frames_processed': frames_processed,
            'total_frames':     total_frames,
            'video_path':       VIDEO_PATH,
            'git_commit':       GIT_COMMIT,
            'git_branch':       GIT_BRANCH,
        }, _f)

    n_dets = sum(len(v) for v in tracks.values())
    print(f'저장 완료: {out_path}')
    print(f'  트랙 수    : {len(tracks)}')
    print(f'  총 탐지    : {n_dets}')
    print(f'  처리 프레임: {frames_processed} / {total_frames}')
    print(f'  git commit : {GIT_COMMIT}')
